In [1]:
# It reads the saved credit-spread ranking, drops anything flagged for gap risk, 
# takes the top 10/15(choice) by score, 
# then force-adds NIFTY/BANKNIFTY (same gap-risk filter applied to them too), 
# and merges into one deduplicated final_candidates list. No API calls here — pure filtering on the 
# last saved ranking file.


import sys
sys.path.append("..")

import pandas as pd
from src.daily_screener import screen_list

credit_spread_ranking = pd.read_excel("../exports/RANKING_credit_spreads.xlsx")

# Keep only gap-risk-clean stocks, take the top 15
clean_candidates = credit_spread_ranking[
    credit_spread_ranking["gap_risk_flag"].isna() | (credit_spread_ranking["gap_risk_flag"] == "")
]
top_15 = clean_candidates.head(15)

# Always include NIFTY and BANKNIFTY, gap-risk-checked the same way
must_include = credit_spread_ranking[credit_spread_ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
must_include_clean = must_include[must_include["gap_risk_flag"].isna() | (must_include["gap_risk_flag"] == "")]

final_candidates = pd.concat([top_15, must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)
final_candidates

,symbol,credit_spread_score,gap_risk_flag,Total flips (closed),Median duration (days),% completed within a monthly cycle,Median max_adverse_pct,Worst max_adverse_pct (4yr),Worst single gap (4yr),current_trend_age_days,trend_age_note
0,NESTLEIND,91.6,NaN,54,24.2,59.3,2.72,5.96,4.37,35,NaN
1,ITC,86.8,NaN,52,23.7,61.5,2.41,9.56,4.68,9,NaN
2,SUNPHARMA,83.7,NaN,53,20.0,66.0,2.57,7.31,3.53,2,NaN
3,COLPAL,81.0,NaN,59,19.2,66.1,3.12,5.84,2.32,1,NaN
4,DRREDDY,81.0,NaN,57,22.2,68.4,2.84,7.88,4.27,27,NaN
5,ULTRACEMCO,80.2,NaN,66,19.0,66.7,2.39,7.21,3.61,51,"note: aged trend — tested data shows momentum,..."
6,TECHM,78.7,NaN,64,19.6,62.5,3.16,6.85,4.37,33,NaN
7,HAVELLS,78.6,NaN,62,21.6,67.7,3.32,6.58,4.42,12,NaN
8,ICICIPRULI,78.5,NaN,58,21.6,65.5,3.18,8.00,4.94,26,NaN
9,PIIND,78.4,NaN,50,22.1,64.0,3.50,8.76,4.07,16,NaN


In [2]:
# For each of the ~16 symbols it looks up the right instrument 
# token (handling NIFTY/BANKNIFTY specially since indices live in a different token space), 
# fetches only the recent catch-up window of 4H candles from Kite (not the full 2022-to-now
# history - process_stock_incremental() merges this onto existing saved data instead of
# replacing it, same fix already applied to the debit spread/option buying/pledged pipelines,
# here for efficiency rather than a data-loss bug),
# reruns Supertrend → flip detection → monthly range → summary stats on the FULL merged
# history, and updates that symbol's _full_report.xlsx. This is the only block that talks to the API

from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token
from src.scale_pipeline import process_stock_incremental
from datetime import datetime

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)

symbols_to_refresh = final_candidates["symbol"].tolist()
index_symbols = {"NIFTY": ("NIFTY 50", "NSE"), "BANKNIFTY": ("NIFTY BANK", "NSE")}

for i, symbol in enumerate(symbols_to_refresh, start=1):
    try:
        if symbol in index_symbols:
            tradingsymbol, exchange = index_symbols[symbol]
            token = get_index_token(kite, tradingsymbol, exchange)
        else:
            token = token_lookup.get(symbol)

        if token is None:
            print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — no token found, skipping")
            continue

        process_stock_incremental(symbol, token, kite, to_date=datetime.now())
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — refreshed")
    except Exception as e:
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — FAILED: {e}")

Fetching 2026-04-08 to 2026-08-07...
Saved 2266 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/NESTLEIND_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/NESTLEIND_full_report.xlsx
[1/16] NESTLEIND — refreshed
Fetching 2026-04-08 to 2026-08-07...
Saved 2266 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/ITC_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/ITC_full_report.xlsx
[2/16] ITC — refreshed
Fetching 2026-04-08 to 2026-08-07...
Saved 2266 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/SUNPHARMA_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_wo

In [3]:
# reads the just-refreshed reports and reports current state: 
# for each symbol, current trend direction, how many days the open trend has run, 
# whether that age has crossed the stock's own median/75th/90th percentile duration,
# and (new) runway_left_vs_median/p75/p90 - the same comparison as a signed number of
# days instead of three separate booleans (negative = already past that checkpoint) -
# sorted so the youngest (most recently flipped) trends show first. Same caveat as
# before: crossing a checkpoint is NOT a reversal/exit signal (tested, see README) -
# read runway as "how this compares to history," not as a warning.


from src.daily_screener import screen_list

status = screen_list(symbols_to_refresh)
status.sort_values("trend_age_days")

,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,runway_left_vs_median,runway_left_vs_p75,runway_left_vs_p90
3,COLPAL,DOWN,2,19.2,False,False,False,17.2,33.0,59.0
2,SUNPHARMA,DOWN,3,20.0,False,False,False,17.0,37.0,62.4
13,BANKNIFTY,UP,3,19.6,False,False,False,16.6,27.8,35.3
14,TORNTPHARM,UP,6,23.1,False,False,False,17.1,37.2,69.2
15,NIFTY,UP,8,15.2,False,False,False,7.2,15.7,38.5
1,ITC,UP,10,23.7,False,False,False,13.7,31.2,53.1
7,HAVELLS,UP,13,21.6,False,False,False,8.6,22.9,41.0
9,PIIND,UP,17,22.1,False,False,False,5.1,24.4,63.3
12,HDFCBANK,DOWN,17,17.2,False,False,False,0.2,19.4,37.6
8,ICICIPRULI,UP,27,21.6,True,False,False,-5.4,8.0,25.6
